In [36]:
# Cell 1: Imports
# Standard libraries for data manipulation and file path handling

import pandas as pd
import os
from pathlib import Path

In [37]:
# Cell 2: Define the clean_season() function
#
# This function reads a raw NHL season CSV from to_clean/, applies all
# necessary transformations, and returns a cleaned DataFrame ready for saving.
#
# Raw file quirks handled here:
#   - Two header rows: row 0 is a group label row, row 1 has actual column names
#   - Column index 2 is the team abbreviation, but its name varies by season
#     ('Team' in some files, 'Unnamed: 2' in others)
#   - Column index 3 (always 'Unnamed: 3') contains '@' for away games, NaN for home
#   - OT column: NaN = regulation, 'OT' = overtime, 'SO' = shootout
#   - Rows where Rslt is NaN are upcoming/unplayed games — these are kept

def clean_season(season: str) -> pd.DataFrame:
    """
    Read and clean a raw NHL season CSV file.

    Parameters
    ----------
    season : str
        Season identifier, e.g. '2021-22'

    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame containing only home games, with standardised
        column names and an 'overtime_type' column.
    """

    # --- 1. Read raw file -------------------------------------------------
    raw_path = Path('to_clean') / f'NHL_{season}_season.csv'

    # header=1 skips the first group-label row and uses row 1 as column names
    df = pd.read_csv(raw_path, header=1)

    rows_in = len(df)

    # --- 2. Identify the team column --------------------------------------
    # Column at index 2 is always the team abbreviation, but its header text
    # varies between seasons.  Rename it to a stable name before filtering.
    team_col_name = df.columns[2]   # 'Team' or 'Unnamed: 2'
    df.rename(columns={team_col_name: 'team'}, inplace=True)

    # --- 3. Filter to home games only ------------------------------------
    # Column index 3 ('Unnamed: 3') contains '@' for away games and NaN for
    # home games.  We keep rows where that column is NOT '@'.
    home_away_col = df.columns[3]   # always 'Unnamed: 3' after header=1
    df = df[df[home_away_col] != '@'].copy()

    home_rows = len(df)

    # --- 4. Add home_away indicator column --------------------------------
    # After filtering, every remaining row is a home game.
    df['home_away'] = 'home'

    # --- 5. Map overtime_type ---------------------------------------------
    # NaN (regulation) -> 'REG', 'OT' -> 'OT', 'SO' -> 'SO'
    # IMPORTANT: rows with NaN in Rslt are upcoming games — for those rows
    # we keep overtime_type as NaN rather than labelling them 'REG'.
    def map_overtime(row):
        # If the game hasn't been played yet, leave overtime_type blank
        if pd.isna(row['Rslt']):
            return float('nan')
        ot_val = row['OT']
        if pd.isna(ot_val):
            return 'REG'
        return str(ot_val).strip()  # 'OT' or 'SO'

    df['overtime_type'] = df.apply(map_overtime, axis=1)

    # --- 6. Parse dates ---------------------------------------------------
    # Convert the Date column to proper YYYY-MM-DD strings
    df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

    # --- 7. Count upcoming rows ------------------------------------------
    upcoming_rows = df['Rslt'].isna().sum()

    # --- 8. Select and rename columns to final schema --------------------
    # Build the output DataFrame using the exact column order required.
    cleaned = pd.DataFrame({
        'game_num':           df['Gtm'],
        'date':               df['Date'],
        'team':               df['team'],
        'home_away':          df['home_away'],
        'opponent':           df['Opp'],
        'result':             df['Rslt'],
        'goals_for':          df['GF'],
        'goals_against':      df['GA'],
        'overtime_type':      df['overtime_type'],
        'team_sog':           df['SOG'],
        'team_pim':           df['PIM'],
        'team_ppg':           df['PPG'],
        'team_ppo':           df['PPO'],
        'team_shg':           df['SHG'],
        'opp_sog':            df['SOG.1'],
        'opp_pim':            df['PIM.1'],
        'opp_ppg':            df['PPG.1'],
        'opp_ppo':            df['PPO.1'],
        'opp_shg':            df['SHG.1'],
        'faceoff_wins':       df['FOW'],
        'faceoff_losses':     df['FOL'],
        'faceoff_pct':        df['FO%'],
        'corsi_for':          df['CF'],
        'corsi_against':      df['CA'],
        'corsi_pct':          df['CF%'],
        'fenwick_for':        df['FF'],
        'fenwick_against':    df['FA'],
        'fenwick_pct':        df['FF%'],
        'off_zone_start_pct': df['oZS%'],
        'pdo':                df['PDO'],
    })

    # Reset index so row numbers are clean (original index carries away-game gaps)
    cleaned.reset_index(drop=True, inplace=True)

    # Return the cleaned DataFrame along with summary counts
    return cleaned, rows_in, home_rows, upcoming_rows

In [38]:
# Cell 3: Process all seasons and print a summary
#
# We iterate over every season, call clean_season(), store the result in a
# dictionary, and print a brief summary showing:
#   - total rows read from the raw file
#   - home-game rows kept after filtering
#   - upcoming (unplayed) rows retained in the output

SEASONS = ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

# Dictionary to hold cleaned DataFrames keyed by season string
cleaned_dfs = {}

print(f"{'Season':<12} {'Rows in':>10} {'Home rows':>10} {'Upcoming':>10}")
print('-' * 46)

for season in SEASONS:
    df_clean, rows_in, home_rows, upcoming = clean_season(season)
    cleaned_dfs[season] = df_clean
    print(f"{season:<12} {rows_in:>10} {home_rows:>10} {upcoming:>10}")

Season          Rows in  Home rows   Upcoming
----------------------------------------------
2021-22            2624       1312          0
2022-23            2624       1312          0
2023-24            2624       1312          0
2024-25            2624       1312          0
2025-26            2624       1312          6


In [39]:
# Cell 4: Validation — compare newly cleaned files against existing data/ files
#
# For each season, we load the existing cleaned CSV from data/ and compare it
# to the freshly produced DataFrame on:
#   - Row count match
#   - Column names and order
#   - Any data differences (using pd.DataFrame.equals for a quick check)
#
# Differences are printed so they can be reviewed before overwriting.

print("Validation: newly cleaned vs. existing data/ files\n")
print(f"{'Season':<12} {'New rows':>10} {'Old rows':>10} {'Cols match':>12} {'Data match':>12}")
print('-' * 60)

for season in SEASONS:
    existing_path = Path('data') / f'NHL_{season}_cleaned.csv'

    # Load existing file if it exists
    if not existing_path.exists():
        print(f"{season:<12}  [no existing file found — will create]")
        continue

    df_existing = pd.read_csv(existing_path)
    df_new      = cleaned_dfs[season]

    new_rows  = len(df_new)
    old_rows  = len(df_existing)

    # Column comparison: names and order
    cols_match = list(df_new.columns) == list(df_existing.columns)

    # Data comparison: only meaningful when row counts and columns both match
    if cols_match and new_rows == old_rows:
        # Compare values; allow NaN == NaN so unplayed rows don't flag as diffs
        data_match = df_new.equals(df_existing)
    else:
        data_match = False

    print(f"{season:<12} {new_rows:>10} {old_rows:>10} {str(cols_match):>12} {str(data_match):>12}")

    # If columns don't match, show which columns differ
    if not cols_match:
        missing_in_new  = set(df_existing.columns) - set(df_new.columns)
        extra_in_new    = set(df_new.columns)      - set(df_existing.columns)
        if missing_in_new:
            print(f"  Columns in existing but NOT in new : {sorted(missing_in_new)}")
        if extra_in_new:
            print(f"  Columns in new but NOT in existing : {sorted(extra_in_new)}")
        if list(df_new.columns) != list(df_existing.columns) and not missing_in_new and not extra_in_new:
            print(f"  Same columns but different order.")
            print(f"  New      order: {list(df_new.columns)}")
            print(f"  Existing order: {list(df_existing.columns)}")

    # If row counts differ, show how many rows diverge
    if new_rows != old_rows:
        print(f"  Row count difference: new={new_rows}, existing={old_rows} (delta={new_rows - old_rows:+d})")

    # If data doesn't match despite same shape, show the first few differing rows
    if not data_match and cols_match and new_rows == old_rows:
        # Find rows where any value differs (treating NaN as equal)
        diff_mask = ~(
            (df_new == df_existing) |
            (df_new.isna() & df_existing.isna())
        ).all(axis=1)
        n_diff = diff_mask.sum()
        print(f"  {n_diff} row(s) have at least one differing value.")
        if n_diff > 0:
            print("  First differing rows (new values):")
            print(df_new[diff_mask].head(3).to_string(index=False))

print("\nValidation complete.")

Validation: newly cleaned vs. existing data/ files

Season         New rows   Old rows   Cols match   Data match
------------------------------------------------------------
2021-22            1312       1312         True         True
2022-23            1312       1312         True         True
2023-24            1312       1312         True         True
2024-25            1312       1312         True         True
2025-26            1312       1312         True        False
  6 row(s) have at least one differing value.
  First differing rows (new values):
 game_num       date team home_away opponent result  goals_for  goals_against overtime_type  team_sog  team_pim  team_ppg  team_ppo  team_shg  opp_sog  opp_pim  opp_ppg  opp_ppo  opp_shg  faceoff_wins  faceoff_losses  faceoff_pct  corsi_for  corsi_against  corsi_pct  fenwick_for  fenwick_against  fenwick_pct  off_zone_start_pct   pdo
       82 2026-04-15  BUF      home      DAL      L        3.0            4.0            SO      24.0 

In [40]:
# Cell 5: Save all cleaned DataFrames to data/ (overwrite existing files)
#
# We write each cleaned DataFrame as a CSV without the pandas row index,
# matching the format of the existing files in data/.

output_dir = Path('data')
output_dir.mkdir(exist_ok=True)  # Create data/ if it somehow doesn't exist

print("Saving cleaned files to data/...\n")

for season, df_clean in cleaned_dfs.items():
    out_path = output_dir / f'NHL_{season}_cleaned.csv'
    # index=False ensures no extra index column is written to the CSV
    df_clean.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}  ({len(df_clean)} rows, {len(df_clean.columns)} columns)")

print("\nAll seasons saved successfully.")

Saving cleaned files to data/...

  Saved: data/NHL_2021-22_cleaned.csv  (1312 rows, 30 columns)
  Saved: data/NHL_2022-23_cleaned.csv  (1312 rows, 30 columns)
  Saved: data/NHL_2023-24_cleaned.csv  (1312 rows, 30 columns)
  Saved: data/NHL_2024-25_cleaned.csv  (1312 rows, 30 columns)
  Saved: data/NHL_2025-26_cleaned.csv  (1312 rows, 30 columns)

All seasons saved successfully.
